# 06K — Training on Kaggle, across sessions

**What this notebook does.** Trains the boundary U-Net on one fold, exactly as
`06_train.ipynb` does — same `src.train.Trainer`, same loss, same fold
protocol, same metrics — but arranged so a run *longer than one Kaggle session*
survives. It stages a checkpoint out of an attached input dataset, resumes from
it, trains until the session budget is nearly spent, stops cleanly on an epoch
boundary, and tells you exactly what to save.

**Why a separate notebook.** On Colab `PERSISTENT_DIR` is a Drive folder and
"write `last.pt` every epoch" is genuinely enough. On Kaggle `PERSISTENT_DIR`
is `/kaggle/working`, which is **deleted when the kernel stops** and reaches
the next session only as a saved version's output, re-attached read-only under
`/kaggle/input` — which is not where `Trainer` looks. That gap needs code, and
the code lives in `src/kaggle_persist.py` on this branch.

> **On Colab, use `06_train.ipynb` instead.** This notebook stops with a clear
> message if the platform is not Kaggle.

**What must already exist.**

- `GH_TOKEN` in **Add-ons → Secrets**, attached to this notebook
- **Internet ON** for this notebook (Settings → Internet). The bootstrap clones
  from GitHub and pip-installs; both fail silently-looking without it.
- the two input datasets attached — the micrographs
  (`session.kaggle.dataset_slug`) and the boundary maps
  (`session.kaggle.gt_dataset_slug`). The boundary maps are produced by
  `02_boundary_gt.ipynb` **on Colab** and uploaded; a Kaggle session cannot
  make them, because `/kaggle/input` is read-only.
- `reports/manifests/<fold>.csv` and `configs/fold_stats.yaml` — `03_tiling.ipynb`
- `configs/dataloader.yaml` with a **kaggle** entry — `04_dataset.ipynb` run here
- `configs/default.yaml` with `model:`, `loss:`, `train.batch_size` — `05_model_and_loss.ipynb`
- *(second session onward)* the previous session's saved output attached as an
  input dataset

**What it produces.** `/kaggle/working/checkpoints/<fold>/{last,best}.pt`,
`reports/train_<fold>.{json,md}` pushed to the repo, and a printed survival
report saying whether the run is finished or needs another session.

**Expected runtime.** One session: whatever `session_budget_hours` allows
(8.5 h by default, minus a 25 min reserve). At ~2.5 min/epoch for `dev` that is
most of the 40-epoch run in one sitting; a slower fold takes two.

# How a multi-session run works here

Four things have to line up, and only the first is already true on `main`:

1. **Every epoch is checkpointed atomically.** `src/train.py` writes `last.pt`
   after each epoch to a temp file, `fsync`s, then renames. A kernel killed
   mid-write leaves the *previous* epoch intact, never a truncated file.
2. **The run stops before the kernel is killed.** `kaggle_persist.budget_guard`
   is passed to `fit(on_epoch_end=...)`. `fit()` calls `save()` *before* the
   callback, so raising there always leaves a complete checkpoint. It stops
   when the time left is less than the **slowest** epoch seen so far — not the
   mean, because one slow epoch at the end is exactly the case that would run
   past the deadline.
3. **You save the version.** A notebook cannot do this for itself. It is a UI
   action: **File → Save Version → Quick Save**, output enabled.
4. **The next session attaches that output** and
   `kaggle_persist.stage_resume()` copies `last.pt` out of `/kaggle/input` into
   `/kaggle/working/checkpoints/<fold>/`, where the ordinary
   `Trainer.maybe_resume()` finds it and re-checks it.

Nothing in step 4 weakens a guard. The staged file is verified to be the right
fold before it is copied and again after, and `maybe_resume()` still refuses it
on a fold or config-hash mismatch. This code moves files; it never decides that
a resume is legitimate.

## Cell 1 — the standard bootstrap block

Identical in every notebook, and copied verbatim from `06_train.ipynb`. Reads
`GH_TOKEN` from the Kaggle secret store, fetches `scripts/bootstrap_session.py`
through the GitHub API, then hands over to `bootstrap()`.

`BRANCH` says `main` on purpose and is not a mistake to fix: the bootstrap
clones `main`, reads `session.kaggle.branch` from the config, and **switches
this checkout to `kaggle` by itself**. That is why the notebooks stay identical
across branches. The session summary it prints will say `(kaggle)`.

In [ ]:
# --- standard bootstrap block: identical in every notebook ---------------
OWNER, REPO, BRANCH = "arhorri", "boundary", "main"

import importlib, os, pathlib, sys, urllib.request


def _gh_token():
    """Read GH_TOKEN from whichever secret store this host provides."""
    try:
        from google.colab import userdata

        return userdata.get("GH_TOKEN")
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        pass
    return os.environ.get("GH_TOKEN")


_token = _gh_token()
if not _token:
    raise SystemExit(
        "GH_TOKEN secret is missing.\n"
        "  Colab : key icon in the left sidebar -> add GH_TOKEN -> notebook access ON\n"
        "  Kaggle: Add-ons -> Secrets -> add GH_TOKEN -> attach to this notebook"
    )

_req = urllib.request.Request(
    f"https://api.github.com/repos/{OWNER}/{REPO}/contents/scripts/bootstrap_session.py?ref={BRANCH}",
    headers={
        "Authorization": f"Bearer {_token}",
        "Accept": "application/vnd.github.raw",
    },
)
pathlib.Path("bootstrap_session.py").write_bytes(urllib.request.urlopen(_req).read())
del _token

if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))
import bootstrap_session

bootstrap_session = importlib.reload(bootstrap_session)

PATHS = bootstrap_session.bootstrap(
    repo_url=f"https://github.com/{OWNER}/{REPO}.git", branch=BRANCH
)

## Confirm the platform, and start the session clock

The budget clock starts **here**, not at the first epoch, and that is the whole
point of putting this cell early. The clone, the pip install and the tile-cache
build all spend session time that the last epoch will not have. A budget
measured from the first epoch would overrun by exactly the setup cost.

This cell also refuses to continue on Colab. Running the Kaggle persistence
path against a Drive `PERSISTENT_DIR` would stage checkpoints that were never
in danger and stop a run that had no deadline.

In [ ]:
from pathlib import Path

from src import kaggle_persist as kp

if PATHS["platform"] != "kaggle":
    raise SystemExit(
        f"This notebook is Kaggle-only; the platform detected as "
        f"{PATHS['platform']!r}.\n"
        "On Colab, PERSISTENT_DIR is a Drive folder that survives on its own "
        "-- run notebooks/06_train.ipynb instead."
    )

persist_cfg = kp.settings()
budget = kp.SessionBudget()

print(f"platform : {PATHS['platform']}")
print(f"GPU      : {PATHS['gpu']}  ({PATHS['vram']})")
print(f"working  : {PATHS['persistent_dir']}   <- DELETED when this kernel stops")
print(f"data     : {PATHS['data_root']}")
print(f"gt maps  : {PATHS['gt_boundaries_root']}")
print(f"\n{budget.describe()}")
print(f"  session_budget_hours {persist_cfg['session_budget_hours']}  "
      f"reserve_minutes {persist_cfg['reserve_minutes']}")
print("  (both from configs/kaggle.yaml session.kaggle.persist)")

print("\nattached input datasets:")
for root in kp.input_roots():
    print(f"  {root}")

## Choose the fold

`FOLD` is the only thing to change. Everything else follows from it: which
manifest is read, which dataset is held out, which `pos_weight` the loss
carries, which sampler weights the loader uses, and where the checkpoints go.

`dev` is the alias for `fold_uhcs2` — same protocol, smallest validation set,
fastest to iterate on. Nothing below is a literal: every value is printed with
the file it came from, because steps 3–5 exist precisely so that these are
measured once and read afterwards.

`EPOCHS` stays `None`. The **total** length of the run is `train.epochs` from
`configs/default.yaml`; how much of it *this session* gets through is decided
by the budget, not by a number typed here. Setting `EPOCHS` to a small value
would silently redefine the run rather than split it.

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from src import dataset as ds
from src import losses as losses_mod
from src import model as model_mod
from src import train as train_mod

FOLD = "dev"          # dev | fold_MetalDam | fold_uhcs1 | fold_uhcs2
EPOCHS = None         # None -> train.epochs; the BUDGET decides this session


def progress(seq, desc=""):
    return tqdm(seq, desc=desc, leave=False)


train_settings = train_mod.load_config()
model_settings = model_mod.load_config()
loss_settings = losses_mod.load_config()
ds_settings = ds.load_config()
fold_stats = ds.load_fold_stats()
entry = fold_stats["folds"][FOLD]

trainer = train_mod.Trainer(fold=FOLD, resolved=PATHS, settings=train_settings,
                            model_settings=model_settings,
                            loss_settings=loss_settings,
                            dataset_settings=ds_settings)

print(f"fold {FOLD}  (alias of {entry.get('alias_of') or '-'}, "
      f"held out: {entry['held_out']})")
print(f"  train {entry['n_train_tiles']} tiles / {entry['n_train_parents']} parents")
print(f"  val   {entry['n_val_tiles']} tiles / {entry['n_val_parents']} parents")
print(f"  pos_weight {entry['pos_weight']}   <- configs/fold_stats.yaml")
print(f"  batch_size {train_settings['batch_size']}   <- configs/default.yaml")
print(f"  epochs (whole run) {train_settings['epochs']}")
print(f"  config hash {trainer.hash}")
print(f"\npersist_cache is {ds_settings['persist_cache']} on this host "
      "(configs/kaggle.yaml): /kaggle/input is local SSD, so the on-disk tile "
      "cache buys nothing and would bloat the saved version.")

## Stage a checkpoint out of the attached inputs

This is the cell that makes session 2 a continuation rather than a restart. It
searches every attached dataset for `checkpoints/<FOLD>/last.pt` (and `best.pt`),
reads each candidate's header, and copies the newest one **whose fold matches**
into `/kaggle/working/checkpoints/<FOLD>/`.

Read the output rather than skimming it:

- **`no attached checkpoint for this fold`** is correct for the *first* session
  and wrong for any later one. If you saved a version last time and still see
  this, the output was not attached as an input — fix that before spending a
  session training from epoch 0 again.
- **`ignoring ...: belongs to fold X`** means a checkpoint is attached but for a
  different fold. It is reported rather than dropped, because a present-but-
  unusable checkpoint is exactly what you need told to you now.
- **`local checkpoint already present`** means `/kaggle/working` already holds
  one — a re-run of this cell inside the same session. Nothing is staged, and
  that is deliberate: the local file is the run in progress, and replacing it
  with an older attached copy would throw away finished epochs.

In [ ]:
staging = kp.stage_resume(FOLD, PATHS)

candidates = pd.DataFrame(staging["candidates"])
if not candidates.empty:
    columns = [c for c in ("source_dataset", "fold", "epoch", "config_hash",
                           "usable", "reason", "path")
               if c in candidates.columns]
    print("\nevery checkpoint found across the attached datasets:")
    print(candidates[columns].to_string(index=False))
else:
    print("\nno checkpoint files matched in any attached dataset")

for item in staging["staged"]:
    print(f"\nstaged {item['name']} from {item['source_dataset']}: "
          f"epoch {item['epoch']}, hash {item['config_hash']}, "
          f"{item['size_mb']} MB")
print(f"\nresumable: {staging['resumable']}")

## Build the loaders, and check the alignment before spending GPU hours

Two confirmations, both cheap now and expensive later.

**The tile cache.** On this host `persist_cache` is off, so a `COLD` build is
expected and costs one decode pass over the source images from `/kaggle/input`
— a local SSD, not Drive. It is charged to the session budget, which is why the
clock started before it.

**The alignment.** Spatial transforms hit image and mask as one `Compose`;
photometric transforms hit the image only. Step 4 tested that. It is still
worth looking at four augmented pairs, because broken alignment produces a
model that trains, converges, and is worthless — and you would not find out for
hours.

In [ ]:
summary = trainer.setup(progress=progress)

print(f"device {summary['device']}  {summary['gpu'] or ''}  AMP {summary['amp']}")
print(f"steps/epoch {summary['steps_per_epoch']}  batch {summary['batch_size']}  "
      f"workers {summary['num_workers']}")
print(f"checkpoints -> {summary['checkpoint_dir']}")

for split, info in summary["cache"].items():
    flag = "WARM" if info["source"] == "warm" else "COLD"
    print(f"tile cache {split}: {flag} -- {info['images']} images in "
          f"{info['seconds']:.1f}s")

print(f"\n{budget.describe()}   <- setup is already charged to it")

composition = pd.DataFrame([
    {"split": split, "dataset": name, "tiles": info["tiles"],
     "parents": info["parents"]}
    for split, comp in (("train", summary["train_composition"]),
                        ("val", summary["val_composition"]))
    for name, info in comp.items()])
print("\nsplit composition:")
print(composition.to_string(index=False))
print(f"\nheld-out dataset: {trainer.held_out}")

### Four augmented training pairs

The mask is drawn over the image in red. If the spatial transform were applied
to only one of the two, the red lines would sit beside the features they trace
rather than on them. Cheapest possible check, and the last one before the GPU
time starts.

In [ ]:
import matplotlib.pyplot as plt

picks = np.random.default_rng(0).choice(len(trainer.train_ds), size=4, replace=False)
fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
for ax, index in zip(axes, picks):
    sample = trainer.train_ds[int(index)]
    image, mask = sample["image"][0], sample["mask"][0]
    shown = (image - image.min()) / max(1e-6, float(np.ptp(image)))
    rgb = np.dstack([shown] * 3)
    rgb[..., 0] = np.where(mask > 0, 1.0, rgb[..., 0])
    rgb[..., 1] = np.where(mask > 0, rgb[..., 1] * 0.2, rgb[..., 1])
    rgb[..., 2] = np.where(mask > 0, rgb[..., 2] * 0.2, rgb[..., 2])
    ax.imshow(rgb)
    ax.set_title(f"{sample['dataset']}\n{sample['tile_id']}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Resume

The staged file is now an ordinary local checkpoint, so this is the same
`maybe_resume()` that runs on Colab, with the same two refusals:

- **wrong fold** — resuming a `fold_uhcs1` checkpoint into a `fold_MetalDam`
  run would train under `pos_weight` 15.974 on weights fitted at 7.111 and
  validate against the wrong held-out dataset. Nothing about that raises on its
  own, so it is checked and refused.
- **wrong config hash** — a resume that quietly adopts a new learning rate
  produces a curve whose halves are different experiments. Refused, with a diff
  of the keys that changed.

Both are exceptions, not warnings. A run started before the threshold sweep
existed will be refused here and that is correct: the sweep grid is hashed
because it changes *which checkpoint is best*.

In [ ]:
status = trainer.maybe_resume()

if status["resumed"]:
    print(f"RESUMING from {status['path']}")
    print(f"  {status['reason']}")
    print(f"  {status['epochs_done']} epoch(s) done; continuing at epoch "
          f"{status['start_epoch']}")
    print(f"  RNG restored: {status['rng_restored']}")
    print(f"  best so far: {status['best']}")
else:
    print(f"STARTING CLEAN -- {status['reason']}")
    if staging["resumable"]:
        raise SystemExit(
            "a checkpoint WAS staged but maybe_resume() did not use it. That "
            "is a real inconsistency, not a warm start -- stop and read the "
            "message above before training over it."
        )
    print(f"  checkpoints -> {trainer.checkpoint_dir}")

remaining = int(train_settings["epochs"]) - trainer.start_epoch
print(f"\n{remaining} of {train_settings['epochs']} epochs still to run")
print(budget.describe())

## Train, until the epochs run out or the session does

`fit()` is given the budget guard. After each epoch — *after* the checkpoint is
written — it prints the time left and how many more epochs fit, and raises
`SessionTimeUp` when the next one would not.

`SessionTimeUp` is caught here and is **not a failure**. It means the session
ended tidily on an epoch boundary with a complete `last.pt` on disk. The cells
below run either way, so the tables, the checks and the report are produced for
a partial session exactly as for a finished one.

In [ ]:
def on_epoch_end(record, trainer):
    m = record["metrics"]["per_dataset"].get(trainer.held_out)
    line = (f"epoch {record['epoch']:>3}  "
            f"train {record['train']['total']:.4f}  "
            f"val {record['val']['total']:.4f}  ")
    if m:
        line += (f"{trainer.held_out} dice {m['best']['dice']:.4f} "
                 f"@ {m['best_threshold']:.2f}  "
                 f"(fixed {m['fixed']['dice']:.4f})  ")
    line += f"{record['seconds'] / 60:.1f} min"
    print(line + ("   <- BEST" if record["is_best"] else ""))


stopped_early = False
try:
    trainer.fit(epochs=EPOCHS,
                on_epoch_end=kp.budget_guard(budget, on_epoch_end),
                progress=progress)
except kp.SessionTimeUp as stop:
    stopped_early = True
    print("\n" + "=" * 72)
    print("SESSION BUDGET REACHED -- this is a clean stop, not an error")
    print("=" * 72)
    print(stop)

## The run so far, in two tables

Scrollback dies with the session; the history lives in the checkpoint, so these
rebuild from `trainer.history` and read identically after a resume. On session
2 they cover **both** sessions, because the resumed history came back with the
checkpoint.

Every metric is shown at two operating points. `fixed` is `train.threshold`
(0.5); `best` is the threshold that maximised Dice for that dataset that epoch.
A metric at a fixed threshold measures the model and the operating point
together and reports the sum as if it were the model — at 0.5 on uhcs2 this
model painted 5.8x too much boundary and precision read 0.087, most of which
was the threshold. `best.pt` is selected on the `best` row for the held-out
dataset.

In [ ]:
loss_history = pd.DataFrame([
    {"epoch": r["epoch"],
     **{f"train_{k}": r["train"][k] for k in ("total", "bce", "dice", "cldice")},
     "val_total": r["val"]["total"], "lr": r["lr"],
     "minutes": r["seconds"] / 60}
    for r in trainer.history])
print("loss terms per epoch (both sessions, if this was a resume)")
print(loss_history.to_string(index=False, float_format=lambda v: f"{v:9.4f}"))

metric_history = pd.DataFrame([
    {"epoch": r["epoch"], "dataset": name,
     "held_out": name == trainer.held_out, "at": row,
     "thr": entry_[row]["threshold"],
     **{k: entry_[row][k] for k in train_mod.METRIC_ORDER},
     "true_frac": entry_[row]["true_fraction"],
     "pred_frac": entry_[row]["pred_fraction"]}
    for r in trainer.history
    for name, entry_ in r["metrics"]["per_dataset"].items()
    for row in ("fixed", "best")])
print("\nvalidation metrics per dataset, at both operating points")
print(metric_history.to_string(index=False, float_format=lambda v: f"{v:7.4f}"))

## Checks

Where this step is declared correct or not. The claim being verified is
narrower than `06_train.ipynb`'s, and deliberately so: **this session made
progress and left a checkpoint that the next session can genuinely resume.**

Convergence checks that need a finished run are not asserted here — a
budget-stopped session has not earned them, and pretending otherwise is how a
partial run gets mistaken for a complete one. The full-run assertions live in
`06_train.ipynb`, and the last session of a run reaches them because
`stopped_early` is then `False`.

In [ ]:
checks = []


def check(name, ok, detail=""):
    checks.append((name, bool(ok)))
    print(f"{'PASS' if ok else 'FAIL'}  {name}{'  -- ' + detail if detail else ''}")


history = trainer.history
check("at least one epoch completed this run", len(history) >= 1,
      f"{len(history)} epoch(s) in trainer.history")

# -- the checkpoint is real, complete, and resumable ----------------------
check("last.pt exists", trainer.last_path.is_file(),
      f"{trainer.last_path} "
      f"({trainer.last_path.stat().st_size / 1024 ** 2:.0f} MB)"
      if trainer.last_path.is_file() else "missing")

state = train_mod.load_checkpoint(trainer.last_path)
required = {"model", "optimizer", "scheduler", "scaler", "rng", "epoch",
            "fold", "config_hash", "history"}
check("last.pt carries everything a resume needs", required <= set(state),
      f"missing {sorted(required - set(state))}" if not required <= set(state)
      else f"epoch {state['epoch']}, {len(state['history'])} history records")
check("last.pt is this fold and this config", state["fold"] == FOLD
      and state["config_hash"] == trainer.hash,
      f"fold {state['fold']}, hash {state['config_hash']}")
check("last.pt is at the latest completed epoch",
      state["epoch"] == history[-1]["epoch"],
      f"checkpoint {state['epoch']}, history ends {history[-1]['epoch']}")

# -- it is where Kaggle can still save it --------------------------------
working = Path(PATHS["persistent_dir"]).resolve()
check("checkpoint is inside /kaggle/working (the only savable place)",
      working in trainer.last_path.resolve().parents, str(working))

reload_ok = True
try:
    probe_model = model_mod.build_model(settings=model_settings)
    probe_model.load_state_dict(state["model"])
except Exception as exc:
    reload_ok, reload_detail = False, f"{type(exc).__name__}: {exc}"
else:
    reload_detail = "state_dict loads into a freshly built model"
check("checkpoint weights load into a fresh model", reload_ok, reload_detail)

# -- what a Save Version would actually capture --------------------------
sizes = kp.working_output_mb(PATHS)
ckpt_mb = sizes["entries"].get("checkpoints", 0.0)
check("checkpoints/ is present in the savable output", ckpt_mb > 0,
      f"{ckpt_mb:.0f} MB of {sizes['total_mb']:.0f} MB total under "
      f"{sizes['root']}")
check("the tile cache is not bloating the saved output",
      sizes["entries"].get("tile_cache", 0.0) < 100,
      f"tile_cache {sizes['entries'].get('tile_cache', 0.0):.0f} MB "
      "-- persist_cache is off on Kaggle by configs/kaggle.yaml")

# -- the budget behaved -------------------------------------------------
check("the session stopped on an epoch boundary, not mid-epoch",
      not stopped_early or trainer.last_path.is_file(),
      "stopped early with a complete checkpoint" if stopped_early
      else "ran to the configured epoch count")
check("budget did not overrun", budget.remaining_seconds() > -60,
      budget.describe())

# -- metrics discipline --------------------------------------------------
final = history[-1]
check("per-dataset metrics reported, not just pooled",
      len(final["metrics"]["per_dataset"]) >= 1,
      f"{sorted(final['metrics']['per_dataset'])}")
check("the validation threshold was swept, not fixed",
      len(final["metrics"]["thresholds"]) >= 10,
      f"{len(final['metrics']['thresholds'])} points, fixed reference "
      f"{final['metrics']['fixed_threshold']:.2f}")
check("pixel accuracy is not reported anywhere",
      not any("accuracy" in key
              for e in final["metrics"]["per_dataset"].values()
              for row in ("fixed", "best") for key in e[row]),
      "boundaries are 5-15% of pixels")

failed = [n for n, ok in checks if not ok]
print(f"\n{len(checks) - len(failed)}/{len(checks)} checks passed")
if failed:
    raise AssertionError("failed checks: " + ", ".join(failed))

## Write the report, push it, and save the session

Two different kinds of persistence, and only one of them is automatic.

**The report** goes to git, and that happens in this cell. It is the only
artefact of the run that git ever sees — checkpoints are hundreds of megabytes
and the repo is not a model registry.

**The checkpoint** goes nowhere unless you save a version. The survival report
below prints what is in `/kaggle/working` and how big each part is, then the
exact UI action. Read the `epochs done` line: if it says the run is incomplete,
saving is not optional — it is the run.

Note where the push lands. The bootstrap put this checkout on the `kaggle`
branch, so `push_results` pushes to `origin/kaggle`, not `main`. That is
expected; merge it back when you want the report on `main`.

In [ ]:
from scripts.push_results import push_results

report = kp.survival_report(FOLD, PATHS, trainer=trainer)

md_path, json_path = train_mod.write_report(trainer)
print(f"\nwrote {md_path}")
print(f"wrote {json_path}")

pushed = push_results(
    f"step 6 (kaggle): training run on {trainer.fold}",
    paths=PATHS,
    expect=[md_path, json_path],
)
print(f"\npushed to origin/{PATHS['branch']}: {pushed}")

done = trainer.history[-1]["epoch"] + 1
total = int(train_settings["epochs"])
if done < total:
    print("\n" + "!" * 72)
    print(f"! RUN INCOMPLETE: {done}/{total} epochs.")
    print("! File -> Save Version -> Quick Save (output enabled) NOW, then")
    print("! start the next session, attach this version's output as an input")
    print("! dataset, and re-run this notebook top to bottom.")
    print("!" * 72)
else:
    print(f"\nRUN COMPLETE: {done}/{total} epochs.")
    print(f"Save a version anyway to keep best.pt: {trainer.best_path}")
    print(f"  fold {trainer.fold}, hash {trainer.hash}, epoch "
          f"{trainer.best['epoch']}, {trainer.best['key']} Dice "
          f"{trainer.best['metric']:.4f}")